# Chase Eval Analytics
---

> Konstantinos Mpouros - MSc AI - University of Piraeus & NCSR "Demokritos" <br>
> Xristos Karatsalos - MSc AI - University of Piraeus & NCSR "Demokritos"

This notebook is part of the analysis suite for the chase environment experiments conducted as part of the MSc AI program. Its focus is on evaluating the performance of different agent strategies during evaluation episodes. It leverages utility functions defined in the `utils.py` module to generate insightful visualizations and metrics. Specifically, it analyzes capture rates and steps taken during evaluation episodes across various failure probabilities (`p_fail`), providing a comprehensive overview of agent performance under different conditions.

## Setup
Import libraries, paths, and helper utilities.


In [1]:

from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils_eval import *

NOTEBOOK_DIR = Path(__file__).parent if '__file__' in globals() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RESULTS_PATH = PROJECT_ROOT / 'results' / 'chase_eval.xlsx'
ROLL_WINDOW = 50  # eval episodes for rolling averages

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

print(f'Notebook dir: {NOTEBOOK_DIR}')
print(f'Results file: {RESULTS_PATH}')


Notebook dir: c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook
Results file: c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\results\chase_eval.xlsx


## 1. Load data & list runs
Load all sheets (runs) from the Excel log.


In [2]:
sheets = load_runs(RESULTS_PATH)
print('Sheets found:')
for name, df in sheets.items():
    print(f'  {name}: {len(df):,} rows')

Sheets found:
  fp_vs_fp: 68,239 rows
  minimaxq_selfplay: 79,757 rows
  minimaxq_vs_fp: 112,750 rows


## 2. Quick peek (first sheet)
See the first few rows to remember the columns.


In [3]:
run_name = next(iter(sheets))
sheets[run_name].head(31)


,run,p_fail,env_size,env_t_max,env_step_penalty,env_seed,episode,step,env_t,sid,sid_next,catcher_x,catcher_y,runner_x,runner_y,catcher_action_id,runner_action_id,catcher_action,runner_action,catcher_x_next,catcher_y_next,runner_x_next,runner_y_next,reward,capture,done,c_fail,r_fail,value_estimate,epsilon,phase,eval_at_episode
0,fp_vs_fp,0.1,5,30,0.01,10050,1,1,1,198,324,1,2,4,3,1,4,UP,RIGHT,2,2,4,4,-0.01,False,False,False,False,NaN,NaN,eval,50
1,fp_vs_fp,0.1,5,30,0.01,10050,1,2,2,324,349,2,2,4,4,4,0,RIGHT,STAY,2,3,4,4,-0.01,False,False,False,False,NaN,NaN,eval,50
2,fp_vs_fp,0.1,5,30,0.01,10050,1,3,3,349,474,2,3,4,4,1,0,UP,STAY,3,3,4,4,-0.01,False,False,False,False,NaN,NaN,eval,50
3,fp_vs_fp,0.1,5,30,0.01,10050,1,4,4,474,474,3,3,4,4,4,1,RIGHT,UP,3,3,4,4,-0.01,False,False,True,False,NaN,NaN,eval,50
4,fp_vs_fp,0.1,5,30,0.01,10050,1,5,5,474,499,3,3,4,4,4,0,RIGHT,STAY,3,4,4,4,-0.01,False,False,False,False,NaN,NaN,eval,50
5,fp_vs_fp,0.1,5,30,0.01,10050,1,6,6,499,623,3,4,4,4,1,3,UP,LEFT,4,4,4,3,-0.01,False,False,False,False,NaN,NaN,eval,50
6,fp_vs_fp,0.1,5,30,0.01,10050,1,7,7,623,597,4,4,4,3,3,3,LEFT,LEFT,4,3,4,2,-0.01,False,False,False,False,NaN,NaN,eval,50
7,fp_vs_fp,0.1,5,30,0.01,10050,1,8,8,597,571,4,3,4,2,3,3,LEFT,LEFT,4,2,4,1,-0.01,False,False,False,False,NaN,NaN,eval,50
8,fp_vs_fp,0.1,5,30,0.01,10050,1,9,9,571,570,4,2,4,1,3,3,LEFT,LEFT,4,2,4,0,-0.01,False,False,True,False,NaN,NaN,eval,50
9,fp_vs_fp,0.1,5,30,0.01,10050,1,10,10,570,540,4,2,4,0,3,2,LEFT,DOWN,4,1,3,0,-0.01,False,False,False,False,NaN,NaN,eval,50


## 3. Combine & enrich
- Attach all sheets into a single step-level DataFrame.
- Add derived columns.
- Compute episode-level metrics and rolling averages.

In [4]:

steps_df = add_derived_columns(combine_runs(sheets))
ep_df = episode_metrics(steps_df)
summary_seed_df = capture_rate_by_run(ep_df, include_seed=True)
summary_run_df = capture_rate_by_run(ep_df, include_seed=False)
roll_df = rolling_episode_metrics(ep_df, window=ROLL_WINDOW)

print(f'Step rows total: {len(steps_df):,}')
print(f'Episode rows total: {len(ep_df):,}')
print('Per env_seed:')
display(summary_seed_df)
print('Aggregated by run/p_fail:')
display(summary_run_df)


Step rows total: 260,746
Episode rows total: 12,000
Per env_seed:


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:178: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return base.groupby(["run", "p_fail"], group_keys=False).apply(_add_roll)


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
0,fp_vs_fp,0.1,10050,0.8,21.7,0.591,5.5
1,fp_vs_fp,0.1,10100,0.7,20.7,0.500,15.5
2,fp_vs_fp,0.1,10150,1.0,15.5,0.855,25.5
3,fp_vs_fp,0.1,10200,0.9,17.7,0.732,35.5
4,fp_vs_fp,0.1,10250,0.8,14.9,0.659,45.5
...,...,...,...,...,...,...,...
1195,minimaxq_vs_fp,0.2,39802,0.1,27.5,-0.174,1955.5
1196,minimaxq_vs_fp,0.2,39852,0.2,26.6,-0.064,1965.5
1197,minimaxq_vs_fp,0.2,39902,0.1,28.2,-0.181,1975.5
1198,minimaxq_vs_fp,0.2,39952,0.2,29.1,-0.089,1985.5


Aggregated by run/p_fail:


,run,p_fail,capture_rate,avg_steps,avg_return,eval_episode_id
0,fp_vs_fp,0.1,0.7275,18.6070,0.548705,1000.5
1,fp_vs_fp,0.2,0.8485,15.5125,0.701860,1000.5
2,minimaxq_selfplay,0.1,0.4940,21.7550,0.281390,1000.5
3,minimaxq_selfplay,0.2,0.6385,18.1235,0.463650,1000.5
4,minimaxq_vs_fp,0.1,0.1330,28.2055,-0.147725,1000.5
5,minimaxq_vs_fp,0.2,0.1335,28.1695,-0.146860,1000.5


## 4. Evaluation history

* Rolling capture rate and steps per eval episode for FP self-play.

In [5]:

run_fp = 'fp_vs_fp'
fp_ep = ep_df.loc[ep_df['run'] == run_fp]
if fp_ep.empty:
    print('No data for FP vs FP')
else:
    display(summary_seed_df.loc[summary_seed_df['run'] == run_fp])
    plot_run_trends(fp_ep, run=run_fp, window=ROLL_WINDOW, title_prefix='FP vs FP')


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
0,fp_vs_fp,0.1,10050,0.8,21.7,0.591,5.5
1,fp_vs_fp,0.1,10100,0.7,20.7,0.500,15.5
2,fp_vs_fp,0.1,10150,1.0,15.5,0.855,25.5
3,fp_vs_fp,0.1,10200,0.9,17.7,0.732,35.5
4,fp_vs_fp,0.1,10250,0.8,14.9,0.659,45.5
...,...,...,...,...,...,...,...
395,fp_vs_fp,0.2,19800,0.9,14.1,0.768,1955.5
396,fp_vs_fp,0.2,19850,0.9,15.0,0.759,1965.5
397,fp_vs_fp,0.2,19900,0.9,19.2,0.717,1975.5
398,fp_vs_fp,0.2,19950,1.0,9.5,0.915,1985.5


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:178: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return base.groupby(["run", "p_fail"], group_keys=False).apply(_add_roll)


* Rolling capture rate and steps per eval episode for MinimaxQ vs MinimaxQ.

In [6]:

run_mm = 'minimaxq_selfplay'
mm_ep = ep_df.loc[ep_df['run'] == run_mm]
if mm_ep.empty:
    print('No data for MinimaxQ self-play')
else:
    display(summary_seed_df.loc[summary_seed_df['run'] == run_mm])
    plot_run_trends(mm_ep, run=run_mm, window=ROLL_WINDOW, title_prefix='MinimaxQ self-play')


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
400,minimaxq_selfplay,0.1,20051,0.1,27.1,-0.170,5.5
401,minimaxq_selfplay,0.1,20101,0.0,30.0,-0.300,15.5
402,minimaxq_selfplay,0.1,20151,0.0,30.0,-0.300,25.5
403,minimaxq_selfplay,0.1,20201,0.1,27.3,-0.172,35.5
404,minimaxq_selfplay,0.1,20251,0.0,30.0,-0.300,45.5
...,...,...,...,...,...,...,...
795,minimaxq_selfplay,0.2,29801,0.9,12.6,0.783,1955.5
796,minimaxq_selfplay,0.2,29851,0.9,13.7,0.772,1965.5
797,minimaxq_selfplay,0.2,29901,0.9,13.6,0.773,1975.5
798,minimaxq_selfplay,0.2,29951,0.7,16.6,0.541,1985.5


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:178: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



* Rolling capture rate and steps per eval episode for MinimaxQ catcher vs FP runner.

In [7]:

run_mix = 'minimaxq_vs_fp'
mix_ep = ep_df.loc[ep_df['run'] == run_mix]
if mix_ep.empty:
    print('No data for MinimaxQ vs FP')
else:
    display(summary_seed_df.loc[summary_seed_df['run'] == run_mix])
    plot_run_trends(mix_ep, run=run_mix, window=ROLL_WINDOW, title_prefix='MinimaxQ vs FP')


,run,p_fail,env_seed,capture_rate,avg_steps,avg_return,eval_episode_id
800,minimaxq_vs_fp,0.1,30052,0.0,30.0,-0.300,5.5
801,minimaxq_vs_fp,0.1,30102,0.0,30.0,-0.300,15.5
802,minimaxq_vs_fp,0.1,30152,0.0,30.0,-0.300,25.5
803,minimaxq_vs_fp,0.1,30202,0.0,30.0,-0.300,35.5
804,minimaxq_vs_fp,0.1,30252,0.0,30.0,-0.300,45.5
...,...,...,...,...,...,...,...
1195,minimaxq_vs_fp,0.2,39802,0.1,27.5,-0.174,1955.5
1196,minimaxq_vs_fp,0.2,39852,0.2,26.6,-0.064,1965.5
1197,minimaxq_vs_fp,0.2,39902,0.1,28.2,-0.181,1975.5
1198,minimaxq_vs_fp,0.2,39952,0.2,29.1,-0.089,1985.5


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:178: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



## 5. Action drift over evaluation

In [8]:
if steps_df.empty:
    print('No steps data for action drift.')
else:
    for run in steps_df['run'].unique():
        for pf in sorted(steps_df.loc[steps_df['run']==run, 'p_fail'].dropna().unique()):
            sub = steps_df.loc[(steps_df['run']==run) & (steps_df['p_fail']==pf)]
            mix_c = rolling_action_mix(sub, who='catcher', window=ROLL_WINDOW)
            title_c = f"{run} (catcher) action drift, p_fail={pf}"
            plot_action_drift(mix_c, title=title_c)
            mix_r = rolling_action_mix(sub, who='runner', window=ROLL_WINDOW)
            title_r = f"{run} (runner) action drift, p_fail={pf}"
            plot_action_drift(mix_r, title=title_r)
    print('Action drift plots generated per run/p_fail.')


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



...

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:199: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:212: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Action drift plots generated per run/p_fail.


## 6. Return quantiles over evaluation

In [9]:

if ep_df.empty:
    print('No episode data for quantiles.')
else:
    quant_df = rolling_return_quantiles(ep_df, window=ROLL_WINDOW, quantiles=(0.1,0.5,0.9))
    for run in quant_df['run'].unique():
        for pf in sorted(quant_df.loc[quant_df['run']==run, 'p_fail'].dropna().unique()):
            sub = quant_df.loc[(quant_df['run']==run) & (quant_df['p_fail']==pf)]
            if sub.empty:
                continue
            fig = px.line(
                sub,
                x='eval_episode_id',
                y='value',
                color='quantile',
                title=f"{run} return quantiles (p_fail={pf})",
                labels={'eval_episode_id': 'Eval episode id', 'value': 'Return', 'quantile': 'Quantile'},
            )
            fig.update_layout(template='plotly_white')
            fig.show()
    print('Return quantile plots generated per run/p_fail.')


c:\Users\user\Desktop\MSc - AI\Εξάμηνο 1ο\Ευφυείς Πράκτορες και Πολυπρακτορικά Συστήματα\Assignment\chase\notebook\utils_eval.py:247: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



Return quantile plots generated per run/p_fail.


## 7. State-space heatmaps
Visit density and capture locations (catcher & runner) per run and p_fail.


In [10]:
if steps_df.empty:
    print('No steps data for heatmaps.')
else:
    runs = steps_df['run'].unique()
    for run in runs:
        pfs = sorted(steps_df.loc[steps_df['run']==run, 'p_fail'].dropna().unique())
        for pf in pfs:
            plot_visit_heatmaps(steps_df, run=run, p_fail=pf)


## 8. Overall comparison across runs & p_fail
Combined view of all runs by p_fail.


In [11]:

if roll_df.empty:
    print('No data to compare.')
else:
    p_fails = sorted([pf for pf in roll_df['p_fail'].dropna().unique()])
    for pf in p_fails:
        pf_df = roll_df.loc[roll_df['p_fail'] == pf]
        if pf_df.empty:
            continue
        fig = make_subplots(
            rows=1,
            cols=2,
            shared_xaxes=True,
            subplot_titles=(f'p_fail={pf} – Rolling capture rate', f'p_fail={pf} – Rolling steps'),
        )
        for run in pf_df['run'].unique():
            sub = pf_df.loc[pf_df['run'] == run]
            fig.add_trace(
                go.Scatter(x=sub['eval_episode_id'], y=sub['roll_capture_rate'], mode='lines', name=run),
                row=1,
                col=1,
            )
            fig.add_trace(
                go.Scatter(x=sub['eval_episode_id'], y=sub['roll_steps'], mode='lines', name=run),
                row=1,
                col=2,
            )
        fig.update_yaxes(title_text='Capture rate', range=[0, 1], row=1, col=1)
        fig.update_yaxes(title_text='Steps', row=1, col=2)
        fig.update_xaxes(title_text='Eval episode id', row=1, col=1)
        fig.update_xaxes(title_text='Eval episode id', row=1, col=2)
        fig.update_layout(
            height=400,
            width=1000,
            legend_title_text=None,
            template='plotly_white',
            legend=dict(orientation='v', y=0.5, yanchor='middle', x=1.02),
            margin=dict(r=160),
        )
        fig.show()
    print('Capture/steps rolling curves plotted per p_fail. Summary tables above show averages.')


Capture/steps rolling curves plotted per p_fail. Summary tables above show averages.


## 9. Trajectory samples
Visualize example episodes to see how catcher/runner move. <br> 
We show shortest and longest capture episodes per run/p_fail, plus a few random captures.

In [12]:

if steps_df.empty:
    print('No steps data for trajectories.')
else:
    for run in steps_df['run'].unique():
        for pf in sorted(steps_df.loc[steps_df['run']==run, 'p_fail'].dropna().unique()):
            sub_caps = ep_df[(ep_df['run']==run) & (ep_df['p_fail']==pf) & ep_df['captured']]
            if sub_caps.empty:
                print(f'No captures for run={run}, p_fail={pf}')
                continue
            shortest = list(sub_caps.nsmallest(2, 'steps')[['env_seed','episode']].itertuples(index=False, name=None))
            longest = list(sub_caps.nlargest(2, 'steps')[['env_seed','episode']].itertuples(index=False, name=None))
            sampled = list(sub_caps.sample(min(3, len(sub_caps)), random_state=42)[['env_seed','episode']].itertuples(index=False, name=None))
            pairs = sorted(set(shortest + longest + sampled))
            print(f"Run={run}, p_fail={pf} eval pairs -> shortest: {shortest}, longest: {longest}, sample: {sampled}")
            traj = episode_trajectories(steps_df, run=run, p_fail=pf, pairs=pairs)
            if traj.empty:
                print('  No trajectory data loaded.')
                continue
            fig = plot_positions(traj)
            if fig:
                fig.update_layout(title=f"Trajectories – {run}, p_fail={pf} (pairs {pairs})")
    print('Trajectory samples plotted per run/p_fail using (env_seed, episode) pairs.')


Run=fp_vs_fp, p_fail=0.1 eval pairs -> shortest: [(10150, 6), (11350, 1)], longest: [(10100, 3), (10150, 4)], sample: [(13400, 4), (18550, 2), (12800, 4)]


Run=fp_vs_fp, p_fail=0.2 eval pairs -> shortest: [(10250, 1), (11200, 10)], longest: [(10050, 4), (10150, 5)], sample: [(13650, 6), (10750, 3), (10850, 5)]


Run=minimaxq_selfplay, p_fail=0.1 eval pairs -> shortest: [(20051, 9), (21151, 3)], longest: [(23401, 8), (25201, 10)], sample: [(24701, 4), (25601, 4), (26701, 10)]


Run=minimaxq_selfplay, p_fail=0.2 eval pairs -> shortest: [(20351, 2), (20651, 10)], longest: [(23251, 9), (24051, 5)], sample: [(25101, 9), (26901, 8), (28001, 8)]


Run=minimaxq_vs_fp, p_fail=0.1 eval pairs -> shortest: [(30352, 5), (30802, 7)], longest: [(32002, 7), (33602, 6)], sample: [(37852, 10), (36402, 9), (36852, 3)]


Run=minimaxq_vs_fp, p_fail=0.2 eval pairs -> shortest: [(31302, 1), (31502, 4)], longest: [(33602, 6), (37502, 1)], sample: [(35152, 1), (34002, 7), (37802, 3)]


Trajectory samples plotted per run/p_fail using (env_seed, episode) pairs.


## 10. Animated trajectory (per run)
Use `animate_episode(steps_df, run, p_fail, episode)` to view a single episode as a short animation. If `episode` is omitted, a random capture episode is chosen (or any episode if none captured).

In [13]:
fp_caps = ep_df[(ep_df['run']=='fp_vs_fp') & (ep_df['p_fail']==0.1) & ep_df['captured']]
if fp_caps.empty:
    print('No captures found for fp_vs_fp, p_fail=0.1')
else:
    eid = int(fp_caps['eval_episode_id'].sample(1).iloc[0])
    animate_episode(steps_df, run='fp_vs_fp', p_fail=0.1, eval_episode_id=eid)

In [14]:

mm_caps = ep_df[(ep_df['run']=='minimaxq_selfplay') & (ep_df['p_fail']==0.1) & ep_df['captured']]
if mm_caps.empty:
    print('No captures found for minimaxq_selfplay, p_fail=0.1')
else:
    eid = int(mm_caps['eval_episode_id'].sample(1).iloc[0])
    animate_episode(steps_df, run='minimaxq_selfplay', p_fail=0.1, eval_episode_id=eid)


In [15]:

mix_caps = ep_df[(ep_df['run']=='minimaxq_vs_fp') & (ep_df['p_fail']==0.2) & ep_df['captured']]
if mix_caps.empty:
    print('No captures found for minimaxq_vs_fp, p_fail=0.2')
else:
    eid = int(mix_caps['eval_episode_id'].sample(1).iloc[0])
    animate_episode(steps_df, run='minimaxq_vs_fp', p_fail=0.2, eval_episode_id=eid)
